In [2]:
# Installing all required libraries
!pip install kafka-python pyspark

In [3]:
# Import all required libraries
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, count, avg, corr, unix_timestamp
from pyspark.sql.types import FloatType, IntegerType, TimestampType
from kafka import KafkaProducer, KafkaConsumer
from collections import deque
import json
import threading
import time

# Load CSV using pandas
df_pandas = pd.read_csv("target_data.csv")

# Check first 5 rows
print("Dataset loaded successfully!")
print(f"Total rows: {len(df_pandas)}")
print(f"Total columns: {len(df_pandas.columns)}")
print("\nFirst 5 rows:")
df_pandas.head()

Dataset loaded successfully!
Total rows: 1000
Total columns: 12

First 5 rows:


,Id,order_status,order_products_value,order_freight_value,order_items_qty,order_purchase_timestamp,order_aproved_at,order_delivered_customer_date,customer_city,customer_state,customer_zip_code_prefix,review_score
0,1,delivered,79.00,17.80,1,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-10 21:25:00,Luziania,GO,728,5
1,2,delivered,119.90,27.16,1,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-08-07 15:27:00,Joinville,SC,892,5
2,3,delivered,519.99,41.69,1,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-17 18:06:00,Serra,ES,291,1
3,4,delivered,29.50,17.92,1,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-12-02 00:28:00,RIO DE JANEIRO,RJ,222,4
4,5,delivered,26.77,23.11,1,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-16 18:17:00,Sao Paulo,SP,40,5


In [4]:
# We simulate Kafka using a simple in-memory queue
# (Real Kafka needs a server - this demonstrates the same concept)

from queue import Queue

# This is our Kafka Topic simulation
# Think of it as a pipe between producer and consumer
kafka_topic = Queue()

# ── PRODUCER ──────────────────────────────────────────
def kafka_producer():
    """
    Reads CSV row by row and sends each row
    into the kafka topic (queue)
    """
    print("Producer started - sending rows to Kafka topic...")

    for index, row in df_pandas.iterrows():
        # Convert each row to JSON text (Kafka sends text)
        message = row.to_json()

        # Put message into topic
        kafka_topic.put(message)

    # Signal that producer is done
    kafka_topic.put(None)
    print(f"Producer finished - sent {len(df_pandas)} rows to topic!")

# ── CONSUMER ──────────────────────────────────────────
# This bucket collects all rows coming out of Kafka
collected_data = []

def kafka_consumer():
    """
    Reads messages from kafka topic one by one
    and stores them in collected_data list
    """
    print("Consumer started - waiting for messages...")

    while True:
        # Get next message from topic
        message = kafka_topic.get()

        # None means producer is done
        if message is None:
            break

        # Convert JSON text back to Python dictionary
        row_dict = json.loads(message)
        collected_data.append(row_dict)

    print(f"Consumer finished - received {len(collected_data)} rows!")

# ── RUN BOTH IN PARALLEL ──────────────────────────────
# Consumer starts first (must be ready before producer)
consumer_thread = threading.Thread(target=kafka_consumer)
producer_thread = threading.Thread(target=kafka_producer)

consumer_thread.start()  # Consumer starts first
time.sleep(1)            # Wait 1 second so consumer is ready
producer_thread.start()  # Then producer starts

# Wait for both to finish
consumer_thread.join()
producer_thread.join()

print("\n✅ Kafka simulation complete!")
print(f"Total rows collected: {len(collected_data)}")

Consumer started - waiting for messages...
Producer started - sending rows to Kafka topic...
Producer finished - sent 1000 rows to topic!
Consumer finished - received 1000 rows!

✅ Kafka simulation complete!
Total rows collected: 1000


In [5]:
# Start Spark Session (the engine of PySpark)
spark = SparkSession.builder \
    .appName("TargetDataPipeline") \
    .getOrCreate()

print("✅ Spark Session started!")

# Convert collected_data list into PySpark DataFrame
df_spark = spark.createDataFrame(collected_data)

print("✅ Spark DataFrame created!")
print(f"Total rows: {df_spark.count()}")
print("\nSchema (column names and data types):")
df_spark.printSchema()

✅ Spark Session started!
✅ Spark DataFrame created!
Total rows: 1000

Schema (column names and data types):
root
 |-- Id: long (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- order_aproved_at: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_freight_value: double (nullable = true)
 |-- order_items_qty: long (nullable = true)
 |-- order_products_value: double (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- review_score: long (nullable = true)



In [6]:
from pyspark.sql.functions import to_timestamp, when

# Fix the 3 date columns from string to timestamp
df_spark = df_spark \
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_aproved_at",
                to_timestamp(col("order_aproved_at"))) \
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date")))

print("✅ Data types fixed!")

# Check for null values in each column
print("\nNull values per column:")
from pyspark.sql.functions import isnan, isnull, sum as spark_sum

null_counts = df_spark.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_spark.columns
])
null_counts.show()

# Remove rows where critical columns are null
df_spark = df_spark.dropna(subset=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_products_value"
])

print(f"\n✅ Nulls removed!")
print(f"Rows remaining: {df_spark.count()}")

# Verify final schema
print("\nFinal corrected schema:")
df_spark.printSchema()

✅ Data types fixed!

Null values per column:
+---+-------------+--------------+------------------------+----------------+-----------------------------+-------------------+---------------+--------------------+------------------------+------------+------------+
| Id|customer_city|customer_state|customer_zip_code_prefix|order_aproved_at|order_delivered_customer_date|order_freight_value|order_items_qty|order_products_value|order_purchase_timestamp|order_status|review_score|
+---+-------------+--------------+------------------------+----------------+-----------------------------+-------------------+---------------+--------------------+------------------------+------------+------------+
|  0|            0|             0|                       0|               0|                           22|                  0|              0|                   0|                       0|           0|           0|
+---+-------------+--------------+------------------------+----------------+-------------------

In [7]:
from pyspark.sql.functions import mean, count

# ── Q1: Mean of order_products_value and order_freight_value ──
print("=" * 50)
print("Q1: Mean Values")
print("=" * 50)

df_spark.select(
    mean("order_products_value").alias("avg_product_value"),
    mean("order_freight_value").alias("avg_freight_value")
).show()

# ── Q2: Distribution of order_status ──
print("=" * 50)
print("Q2: Distribution of Order Status")
print("=" * 50)

df_spark.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

Q1: Mean Values
+-----------------+------------------+
|avg_product_value| avg_freight_value|
+-----------------+------------------+
|127.8475153374236|21.405429447852764|
+-----------------+------------------+

Q2: Distribution of Order Status
+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|  961|
|     shipped|   12|
|    canceled|    3|
|    invoiced|    1|
|  processing|    1|
+------------+-----+



In [9]:
# ── Q4: Missing Values Check (Fixed) ──
print("=" * 50)
print("Q4: Missing Values Check")
print("=" * 50)

from pyspark.sql.functions import sum as sql_sum, isnull

null_counts = df_spark.select([
    sql_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_spark.columns
])
null_counts.show()

# Summary message
print("\nConclusion:")
print("We already removed 22 null rows in Block 5")
print("Remaining dataset is clean ✅")

Q4: Missing Values Check
+---+-------------+--------------+------------------------+----------------+-----------------------------+-------------------+---------------+--------------------+------------------------+------------+------------+
| Id|customer_city|customer_state|customer_zip_code_prefix|order_aproved_at|order_delivered_customer_date|order_freight_value|order_items_qty|order_products_value|order_purchase_timestamp|order_status|review_score|
+---+-------------+--------------+------------------------+----------------+-----------------------------+-------------------+---------------+--------------------+------------------------+------------+------------+
|  0|            0|             0|                       0|               0|                            0|                  0|              0|                   0|                       0|           0|           0|
+---+-------------+--------------+------------------------+----------------+-----------------------------+---------

In [10]:
# ── Q5: Top 5 cities with most orders ──
print("=" * 50)
print("Q5: Top 5 Cities With Most Orders")
print("=" * 50)

df_spark.groupBy("customer_city") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(5) \
    .show()

# ── Q6: Percentage of each order status ──
print("=" * 50)
print("Q6: Percentage of Orders by Status")
print("=" * 50)

from pyspark.sql.functions import round as spark_round

total_orders = df_spark.count()

df_spark.groupBy("order_status") \
    .count() \
    .withColumn(
        "percentage",
        spark_round((col("count") / total_orders) * 100, 2)
    ) \
    .orderBy("count", ascending=False) \
    .show()

Q5: Top 5 Cities With Most Orders
+--------------+-----+
| customer_city|count|
+--------------+-----+
|     Sao Paulo|  141|
|RIO DE JANEIRO|   71|
|      BRASILIA|   24|
|Belo Horizonte|   21|
|      Curitiba|   19|
+--------------+-----+

Q6: Percentage of Orders by Status
+------------+-----+----------+
|order_status|count|percentage|
+------------+-----+----------+
|   delivered|  961|     98.26|
|     shipped|   12|      1.23|
|    canceled|    3|      0.31|
|    invoiced|    1|       0.1|
|  processing|    1|       0.1|
+------------+-----+----------+



In [11]:
# ── Processing Q1: Total Sales Per City ──
print("=" * 50)
print("Processing Q1: Total Sales Per City")
print("=" * 50)

from pyspark.sql.functions import sum as sql_sum, round as spark_round

df_spark.groupBy("customer_city") \
    .agg(
        spark_round(sql_sum("order_products_value"), 2)
        .alias("total_sales")
    ) \
    .orderBy("total_sales", ascending=False) \
    .show(10)

Processing Q1: Total Sales Per City
+--------------+-----------+
| customer_city|total_sales|
+--------------+-----------+
|     Sao Paulo|   15542.07|
|RIO DE JANEIRO|   11329.36|
|Belo Horizonte|    3614.87|
|      BRASILIA|    2209.92|
|      Curitiba|    2155.93|
|   Sao Goncalo|    1470.39|
|       Diadema|     1458.9|
|     Cabo Frio|    1440.93|
|   Santo Andre|    1359.21|
|       Limeira|    1286.45|
+--------------+-----------+
only showing top 10 rows


In [12]:
# ── Processing Q2: Correlation ──
print("=" * 50)
print("Processing Q2: Correlations")
print("=" * 50)

# Correlation between order value and freight
corr1 = df_spark.stat.corr("order_products_value", "order_freight_value")
print(f"Correlation (Product Value vs Freight): {round(corr1, 4)}")

# Correlation between order value and item quantity
corr2 = df_spark.stat.corr("order_products_value", "order_items_qty")
print(f"Correlation (Product Value vs Items Qty): {round(corr2, 4)}")

# Correlation between freight and item quantity
corr3 = df_spark.stat.corr("order_freight_value", "order_items_qty")
print(f"Correlation (Freight Value vs Items Qty): {round(corr3, 4)}")

print("\nInterpretation Guide:")
print("0.7 to 1.0  = Strong positive correlation")
print("0.4 to 0.7  = Moderate positive correlation")
print("0.0 to 0.4  = Weak correlation")
print("Negative    = Inverse relationship")

Processing Q2: Correlations
Correlation (Product Value vs Freight): 0.4712
Correlation (Product Value vs Items Qty): 0.2645
Correlation (Freight Value vs Items Qty): 0.6339

Interpretation Guide:
0.7 to 1.0  = Strong positive correlation
0.4 to 0.7  = Moderate positive correlation
0.0 to 0.4  = Weak correlation
Negative    = Inverse relationship


In [13]:
# ── Processing Q3: Average Delivery and Approval Times ──
print("=" * 50)
print("Processing Q3: Average Delivery & Approval Times")
print("=" * 50)

from pyspark.sql.functions import unix_timestamp, round as spark_round, avg

# Calculate delivery time in days
# (delivered date - purchase date)
df_with_times = df_spark.withColumn(
    "delivery_time_days",
    spark_round(
        (unix_timestamp("order_delivered_customer_date") -
         unix_timestamp("order_purchase_timestamp")) / 86400, 2
    )
).withColumn(
    "approval_time_hours",
    spark_round(
        (unix_timestamp("order_aproved_at") -
         unix_timestamp("order_purchase_timestamp")) / 3600, 2
    )
)

# Calculate averages
df_with_times.select(
    avg("delivery_time_days").alias("avg_delivery_days"),
    avg("approval_time_hours").alias("avg_approval_hours")
).show()

Processing Q3: Average Delivery & Approval Times
+------------------+------------------+
| avg_delivery_days|avg_approval_hours|
+------------------+------------------+
|12.327014314928421|10.408865030674837|
+------------------+------------------+



In [14]:
# ── Processing Q4: Average Review Score Per Order ──
print("=" * 50)
print("Processing Q4: Average Review Score")
print("=" * 50)

from pyspark.sql.functions import avg, round as spark_round

# Overall average review score
df_spark.select(
    spark_round(avg("review_score"), 2)
    .alias("avg_review_score")
).show()

# Average review score by order status
print("Average Review Score by Order Status:")
df_spark.groupBy("order_status") \
    .agg(
        spark_round(avg("review_score"), 2)
        .alias("avg_review_score")
    ) \
    .orderBy("avg_review_score", ascending=False) \
    .show()

Processing Q4: Average Review Score
+----------------+
|avg_review_score|
+----------------+
|            4.09|
+----------------+

Average Review Score by Order Status:
+------------+----------------+
|order_status|avg_review_score|
+------------+----------------+
|   delivered|            4.14|
|     shipped|            1.25|
|    canceled|             1.0|
|    invoiced|             1.0|
|  processing|             1.0|
+------------+----------------+



In [15]:
# ── Processing Q5: Top 3 Fastest & Slowest Delivery Cities ──
print("=" * 50)
print("Processing Q5: Fastest & Slowest Delivery Cities")
print("=" * 50)

from pyspark.sql.functions import avg, round as spark_round

# Use df_with_times which already has delivery_time_days column
city_delivery = df_with_times.groupBy("customer_city") \
    .agg(
        spark_round(avg("delivery_time_days"), 2)
        .alias("avg_delivery_days"),
        count("Id").alias("total_orders")
    ) \
    .filter(col("total_orders") >= 5)  # Only cities with 5+ orders

# Top 3 FASTEST cities
print("Top 3 Fastest Delivery Cities:")
city_delivery \
    .orderBy("avg_delivery_days", ascending=True) \
    .show(3)

# Top 3 SLOWEST cities
print("Top 3 Slowest Delivery Cities:")
city_delivery \
    .orderBy("avg_delivery_days", ascending=False) \
    .show(3)

Processing Q5: Fastest & Slowest Delivery Cities
Top 3 Fastest Delivery Cities:
+-------------+-----------------+------------+
|customer_city|avg_delivery_days|total_orders|
+-------------+-----------------+------------+
|       Recife|             7.41|           5|
|      Niteroi|             7.85|           8|
|Volta Redonda|             8.39|           6|
+-------------+-----------------+------------+
only showing top 3 rows
Top 3 Slowest Delivery Cities:
+---------------+-----------------+------------+
|  customer_city|avg_delivery_days|total_orders|
+---------------+-----------------+------------+
|    Nova Iguacu|            18.59|           5|
|        Barueri|             18.0|           8|
|Itaquaquecetuba|            16.19|           6|
+---------------+-----------------+------------+
only showing top 3 rows


In [16]:
# ── Processing Q6: Delivery Time vs Review Score ──
print("=" * 50)
print("Processing Q6: Delivery Time vs Review Score")
print("=" * 50)

from pyspark.sql.functions import avg, round as spark_round, corr

# Average delivery time for each review score
print("Average Delivery Time per Review Score:")
df_with_times.groupBy("review_score") \
    .agg(
        spark_round(avg("delivery_time_days"), 2)
        .alias("avg_delivery_days"),
        count("Id").alias("total_orders")
    ) \
    .orderBy("review_score", ascending=False) \
    .show()

# Correlation between delivery time and review score
correlation = df_with_times.stat.corr(
    "delivery_time_days",
    "review_score"
)
print(f"Correlation (Delivery Time vs Review Score): {round(correlation, 4)}")
print("\nInterpretation:")
print("Negative value = longer delivery = lower review score")

Processing Q6: Delivery Time vs Review Score
Average Delivery Time per Review Score:
+------------+-----------------+------------+
|review_score|avg_delivery_days|total_orders|
+------------+-----------------+------------+
|           5|            12.49|         565|
|           4|            12.06|         193|
|           3|            12.82|          80|
|           2|            11.03|          25|
|           1|            11.92|         115|
+------------+-----------------+------------+

Correlation (Delivery Time vs Review Score): 0.0218

Interpretation:
Negative value = longer delivery = lower review score


In [17]:
# ── Save to Hive Table ──
print("=" * 50)
print("Saving Data to Hive Table")
print("=" * 50)

# Configure Spark to support Hive
spark.sql("CREATE DATABASE IF NOT EXISTS target_db")
spark.sql("USE target_db")

# Save main cleaned dataset
df_spark.write \
    .mode("overwrite") \
    .saveAsTable("target_db.sales_data")

print("✅ Main dataset saved to Hive!")

# Save processed data with delivery times
df_with_times.write \
    .mode("overwrite") \
    .saveAsTable("target_db.sales_with_delivery")

print("✅ Processed dataset saved to Hive!")

# Verify tables were created
print("\nTables in target_db:")
spark.sql("SHOW TABLES IN target_db").show()

Saving Data to Hive Table
✅ Main dataset saved to Hive!
✅ Processed dataset saved to Hive!

Tables in target_db:
+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|target_db|         sales_data|      false|
|target_db|sales_with_delivery|      false|
+---------+-------------------+-----------+

